# 🗓️ 9일차 스터디 노트북 — 오픈 주소법 (선형 탐사법)

**오늘 범위**: 03-4 오픈 주소법 → 재해시(rehashing) → 선형 탐사법(linear probing) → 버킷의 3가지 속성(OCCUPIED/EMPTY/DELETED) → Bucket/OpenHash 클래스 → 체인법 vs 오픈 주소법

*(다음: 04장 스택과 큐)*

## 난이도 태그
- 🟢 **기본** — 전원 필수 (개념을 쓸 수 있는지)
- 🟡 **표준** — 팀 목표선 (왜 그런지 + 변형/디버깅)
- 🔴 **심화** — 도전 (효율·엣지케이스·일반화)

## 유형 태그
[예측] 실행 전 결과 맞히기 · [구현] TODO 채우기 · [디버깅] 버그 찾기 · [설명] 왜인지 서술 · [실험] 직접 찍어보기

## 푸는 법
1. 위에서부터 순서대로 (Remind → 본문 🟢 → 🟡 → 🔴)
2. 코드 셀은 직접 실행해서 확인
3. 막히면 힌트만 보고, 정답은 맨 아래에서
4. 금요일 코딩테스트는 🟢🟡 범위

> 📁 아래 "부록: OpenHash 전체 코드" 셀을 **먼저 실행**하면 `open_hash.py` 없이도 노트북이 단독으로 돌아가.

> ⚠️ **오늘의 하이라이트**: 4번 문제(`[Bucket()] * capacity`)는 5일차 얕은 복사가 실전 지뢰로 등장하는 자리야. 교재 코드에 그대로 들어있는 진짜 시한폭탄이니 꼭 풀어봐.

---

## 🔁 [Remind] 워밍업 — 8일차 복습 (한 단계 업)

8일차 체인법을 확실히 잡고 오픈 주소법으로 넘어가자.

### R-1. 🟡 [설명] 충돌 대처법 2가지, 뭐가 달라?

8일차 교재 133p에 충돌 대처법이 2개 나왔어:
- **체인법**: 해시값이 같은 원소를 연결 리스트로 관리
- **오픈 주소법**: 빈 버킷을 찾을 때까지 해시를 반복 ← 오늘 배운 것

같은 데이터 `1(수연), 5(동혁), 10(예지), 12(원준), 14(민서)`를 `capacity=13`에 넣었을 때 (14와 1이 충돌):

- **체인법**(8일차)에서 14는 어디에 저장됐지?
- **오픈 주소법**(오늘)에서 14는 어디에 저장될까?
- 두 방식의 근본적인 차이를 한 문장으로 요약해봐.

*(힌트: 교재 150p에 두 방식의 실행 결과 비교가 정리돼 있어.)*

*(여기에 답 작성)*

### R-2. 🔴 [디버깅] 체인법 remove의 head 케이스, 또

8일차 6번 문제를 뒤집어봤어. 아래 `remove`는 `pp is None` 분기가 **있는데도** 틀렸어. 어디가 문제일까?

*(힌트: 순서를 잘 봐. `pp`는 언제 갱신되지?)*

In [ ]:
from __future__ import annotations
from typing import Any

class Node:
    def __init__(self, key, value, next):
        self.key, self.value, self.next = key, value, next

class MiniHash:
    def __init__(self, capacity):
        self.capacity = capacity
        self.table = [None] * capacity

    def hash_value(self, key):
        return key % self.capacity

    def add(self, key, value):
        idx = self.hash_value(key)
        self.table[idx] = Node(key, value, self.table[idx])
        return True

    def remove(self, key):
        idx = self.hash_value(key)
        p = self.table[idx]
        pp = None
        while p is not None:
            pp = p              # <- 이 위치가 문제
            if p.key == key:
                if pp is None:
                    self.table[idx] = p.next
                else:
                    pp.next = p.next
                return True
            p = p.next
        return False

    def dump_bucket(self, idx):
        p = self.table[idx]
        out = []
        while p is not None:
            out.append(f'{p.key}({p.value})')
            p = p.next
        return ' -> '.join(out) if out else '(비어있음)'

h = MiniHash(13)
h.add(1, 'A')
h.add(14, 'B')   # 버킷 1: 14 -> 1
print('삭제 전 버킷1:', h.dump_bucket(1))
print('remove(14):', h.remove(14))
print('삭제 후 버킷1:', h.dump_bucket(1), '  <- 14가 지워졌나?')


---
## 📦 부록 — OpenHash 전체 코드

`open_hash.py`가 같은 폴더에 없다면 **이 셀을 먼저 실행**해. (있으면 건너뛰어도 됨)

In [ ]:
from __future__ import annotations
from typing import Any
from enum import Enum
import hashlib

class Status(Enum):
    OCCUPIED = 0   # 데이터를 저장
    EMPTY = 1      # 비어 있음
    DELETED = 2    # 삭제 완료

class Bucket:
    '''해시를 구성하는 버킷'''
    def __init__(self, key: Any = None, value: Any = None,
                 stat: Status = Status.EMPTY) -> None:
        self.key = key
        self.value = value
        self.stat = stat

    def set(self, key: Any, value: Any, stat: Status) -> None:
        '''모든 필드에 값을 설정'''
        self.key = key
        self.value = value
        self.stat = stat

    def set_status(self, stat: Status) -> None:
        '''속성을 설정'''
        self.stat = stat

class OpenHash:
    '''오픈 주소법으로 구현하는 해시 클래스'''
    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.table = [Bucket()] * self.capacity   # 👀 4번 문제에서 다룰 줄

    def hash_value(self, key: Any) -> int:
        '''해시값을 구함'''
        if isinstance(key, int):
            return key % self.capacity
        return (int(hashlib.md5(str(key).encode()).hexdigest(), 16) % self.capacity)

    def rehash_value(self, key: Any) -> int:
        '''재해시값을 구함'''
        return (self.hash_value(key) + 1) % self.capacity

    def search_node(self, key: Any) -> Any:
        '''키가 key인 버킷을 검색'''
        hash = self.hash_value(key)
        p = self.table[hash]
        for i in range(self.capacity):
            if p.stat == Status.EMPTY:
                break
            elif p.stat == Status.OCCUPIED and p.key == key:
                return p
            hash = self.rehash_value(hash)
            p = self.table[hash]
        return None

    def search(self, key: Any) -> Any:
        '''키가 key인 원소를 검색하여 값을 반환'''
        p = self.search_node(key)
        if p is not None:
            return p.value
        else:
            return None

    def add(self, key: Any, value: Any) -> bool:
        '''키가 key이고 값이 value인 원소를 추가'''
        if self.search(key) is not None:
            return False
        hash = self.hash_value(key)
        p = self.table[hash]
        for i in range(self.capacity):
            if p.stat == Status.EMPTY or p.stat == Status.DELETED:
                self.table[hash] = Bucket(key, value, Status.OCCUPIED)
                return True
            hash = self.rehash_value(hash)
            p = self.table[hash]
        return False

    def remove(self, key: Any) -> int:
        '''키가 key인 원소를 삭제'''
        p = self.search_node(key)
        if p is None:
            return False
        p.set_status(Status.DELETED)
        return True

    def dump(self) -> None:
        '''해시 테이블을 덤프'''
        for i in range(self.capacity):
            print(f'{i:2} ', end='')
            if self.table[i].stat == Status.OCCUPIED:
                print(f'{self.table[i].key} ({self.table[i].value})')
            elif self.table[i].stat == Status.EMPTY:
                print('-- 미등록 --')
            elif self.table[i].stat == Status.DELETED:
                print('-- 삭제 완료 --')

print('OpenHash 준비 완료')


---
## 📖 오늘의 핵심 개념

### 오픈 주소법(open addressing)
충돌이 발생했을 때 **재해시(rehashing)**를 수행하여 **빈 버킷을 찾는** 방법. **닫힌 해시법(closed hashing)**이라고도 함.

> ⚠️ 헷갈리기 쉬운 이름: **체인법 = 오픈 해싱(open hashing)**, **오픈 주소법 = 닫힌 해시법(closed hashing)**. 이름이 서로 엇갈려 있으니 주의!

### 재해시 & 선형 탐사법
재해시를 위한 해시 함수는 **자유롭게 정할 수 있어**. 교재는 가장 단순한 걸 씀:

```python
def rehash_value(self, key):
    return (self.hash_value(key) + 1) % self.capacity   # 키에 1을 더해 13으로 나눈 나머지
```

빈 버킷이 나올 때까지 **한 칸씩 옆으로** 재해시를 반복 → **선형 탐사법(linear probing)**

**교재 145p 예시** — 18을 추가할 때 (`18 % 13 = 5`):
1. 버킷 5 → 이미 5가 있음 → 충돌!
2. 재해시 `(18+1) % 13 = 6` → 버킷 6도 6이 있음 → 또 충돌!
3. 재해시 `(19+1) % 13 = 7` → 버킷 7은 비어있음 → **추가 성공**

### 버킷의 3가지 속성 (Status)

| 속성 | 의미 | 교재 기호 |
|---|---|---|
| `OCCUPIED` | 데이터가 저장되어 있음 | 숫자 |
| `EMPTY` | 비어 있음 | `-` |
| `DELETED` | 삭제 완료 | `★` |

**왜 3가지나 필요할까?** (교재 145p) — 5를 삭제할 때 버킷을 그냥 `EMPTY`로 비우면, 해시값이 같은 **18을 검색할 때 "5번 버킷이 비었네 = 없는 데이터군"**이라고 착각해서 검색에 실패해. `DELETED`는 **"여기 있던 건 지워졌지만, 계속 탐사해야 해"**라는 신호야.

| 검색 중 만난 속성 | 어떻게? |
|---|---|
| `EMPTY` | 여기서 멈춤 → 검색 실패 |
| `DELETED` | 그냥 지나쳐서 계속 탐사 |
| `OCCUPIED` + 키 일치 | 검색 성공 |
| `OCCUPIED` + 키 불일치 | 재해시해서 계속 |

### 클래스 구조
- **Status**: `Enum`으로 3가지 속성 정의
- **Bucket**: `key`, `value`, `stat` — 8일차 `Node`와 달리 **`next` 포인터가 없음**! (연결 리스트를 안 쓰니까)
- **OpenHash**: `capacity`, `table`

### 3대 연산
| 연산 | 방식 |
|---|---|
| `search_node` | 해시값 → 버킷 → `EMPTY` 만날 때까지 재해시 반복 |
| `add` | 중복 검사 후, `EMPTY` **또는 `DELETED`** 버킷을 찾아 저장 |
| `remove` | `search_node`로 찾아서 `DELETED`로 표시만 (실제로 안 지움) |

---
## 🟢 기본 문제

### 1. [예측] 재해시 추적 & dump 예측

`OpenHash(13)`에 `1(수연), 5(동혁), 10(예지), 12(원준), 14(민서)`를 순서대로 추가해.

**실행 전에** 손으로 계산해봐:

| 키 | 해시값 | 충돌? | 최종 버킷 |
|---|---|---|---|
| 1 | ? | | ? |
| 5 | ? | | ? |
| 10 | ? | | ? |
| 12 | ? | | ? |
| 14 | ? | ? | ? |

- 14는 어디에 저장될까? 재해시가 몇 번 일어나?
- 8일차 체인법에서는 14가 어디 있었지? 뭐가 달라?

In [ ]:
# 예측 먼저 하고 실행!
h = OpenHash(13)
h.add(1, '수연')
h.add(5, '동혁')
h.add(10, '예지')
h.add(12, '원준')
h.add(14, '민서')
h.dump()


### 2. [구현] rehash_value & search_node 채우기

TODO를 채워봐. `search_node`는 오픈 주소법의 심장이야.

In [ ]:
class MyOpenHash(OpenHash):
    def rehash_value(self, key):
        '''재해시값을 구함 (선형 탐사법: 한 칸 옆으로)'''
        return ___   # TODO

    def search_node(self, key):
        '''키가 key인 버킷을 검색'''
        hash = self.hash_value(key)
        p = self.table[hash]
        for i in range(self.capacity):
            if p.stat == ___:        # TODO: 어떤 속성을 만나면 검색 실패로 끝내지?
                break
            elif p.stat == ___ and p.key == key:   # TODO: 검색 성공 조건
                return p
            hash = self.rehash_value(hash)
            p = self.table[hash]
        return None

h = MyOpenHash(13)
h.add(5, 'five')
h.add(18, 'eighteen')   # 18%13=5 -> 충돌 -> 재해시 -> 6
print(h.search(5))       # five
print(h.search(18))      # eighteen
print(h.search(31))      # None


### 3. [설명] 선형 탐사법이란

- 교재 145p는 재해시를 반복하는 이 방식을 **선형 탐사법(linear probing)**이라고 불러. 왜 "선형"이라는 이름이 붙었을까?
- 교재는 "재해시를 위한 해시 함수는 **자유롭게 정할 수 있습니다**"라고 해. 그럼 `+1` 말고 `+3`으로 해도 될까? 문제가 생긴다면 어떤 경우일까?

*(힌트: `capacity=12`에서 `+3`씩 건너뛰면 어떤 버킷들만 방문하게 될까? 8일차 소수 실험이랑 같은 결이야.)*

*(여기에 답 작성)*

---
## 🟡 표준 문제

### 4. [디버깅] `[Bucket()] * capacity` — 교재에 숨은 시한폭탄 🔥🔥🔥

`OpenHash.__init__`에는 이 줄이 있어 (교재 147p 040행도 **동일**):

```python
self.table = [Bucket()] * self.capacity
```

**(a)** 먼저 아래를 실행해서 `table`의 버킷들이 서로 다른 객체인지 확인해봐.

In [ ]:
h = OpenHash(13)
print('id(table[0]) =', id(h.table[0]))
print('id(table[1]) =', id(h.table[1]))
print('id(table[7]) =', id(h.table[7]))
print()
print('table[0] is table[1] is table[7] ?', h.table[0] is h.table[1] is h.table[7])
print('실제로 존재하는 서로 다른 Bucket 객체 개수:', len(set(id(b) for b in h.table)))
print('capacity =', h.capacity)


**(b)** 그런데 신기하게도 지금 코드는 **정상 동작해.** 왜 우연히 살아있는 걸까?

힌트: `add`의 이 줄을 잘 봐.
```python
self.table[hash] = Bucket(key, value, Status.OCCUPIED)   # 재대입
```

**(c)** `Bucket` 클래스에는 `set()` 메서드가 정의돼 있는데, **코드 어디에서도 안 쓰여**. 만약 `add`에서 재대입 대신 `set()`으로 제자리 수정을 했다면? 예측하고 실행해봐.

In [ ]:
class BombHash(OpenHash):
    def add(self, key, value):
        if self.search(key) is not None:
            return False
        hash = self.hash_value(key)
        p = self.table[hash]
        for i in range(self.capacity):
            if p.stat == Status.EMPTY or p.stat == Status.DELETED:
                p.set(key, value, Status.OCCUPIED)   # <- 재대입 대신 제자리 수정
                return True
            hash = self.rehash_value(hash)
            p = self.table[hash]
        return False

h = BombHash(13)
h.add(1, '수연')
print('키를 딱 1개만 넣었는데 dump 결과:')
h.dump()


**(d)** 이게 5일차에 배운 어떤 개념과 정확히 같은 문제인지 설명하고, `__init__`을 어떻게 고쳐야 안전한지 써봐.

*(여기에 답 작성)*

---
### 5. [예측] DELETED는 왜 필요해?

교재 145p: "인덱스가 5인 버킷을 비우기만 하면 될 것 같지만 실제로는 그렇지 않습니다."

`remove`가 `Status.DELETED` 대신 `Status.EMPTY`로 표시했다면 무슨 일이 생길까? 예측하고 실행해봐.

*(힌트: `search_node`는 `EMPTY`를 만나면 `break`해.)*

In [ ]:
class NoDeletedHash(OpenHash):
    def remove(self, key):
        p = self.search_node(key)
        if p is None:
            return False
        p.set_status(Status.EMPTY)   # <- DELETED 대신 EMPTY
        return True

for cls, name in [(OpenHash, 'DELETED 사용 (원본)'), (NoDeletedHash, 'EMPTY 사용 (버그)')]:
    h = cls(13)
    h.add(5, 'five')
    h.add(18, 'eighteen')   # 18%13=5 -> 충돌 -> 재해시 -> 버킷 6
    print(f'--- {name} ---')
    print('  5 삭제 전 search(18):', h.search(18))
    h.remove(5)
    print('  5 삭제 후 search(18):', h.search(18))
    print()


### 6. [설명] add는 왜 `EMPTY or DELETED` 둘 다 받아?

`search_node`와 `add`가 `DELETED`를 대하는 태도가 정반대야:

```python
# search_node: DELETED를 그냥 지나침 (계속 탐사)
if p.stat == Status.EMPTY:
    break

# add: DELETED도 저장 가능한 자리로 취급
if p.stat == Status.EMPTY or p.stat == Status.DELETED:
    self.table[hash] = Bucket(key, value, Status.OCCUPIED)
```

- 왜 검색은 지나치고 추가는 재사용할까? 각각의 관점에서 `DELETED`가 무슨 의미인지 설명해봐.
- 만약 `add`가 `DELETED`를 재사용하지 않고 `EMPTY`만 찾는다면 어떤 문제가 생길까?

*(여기에 답 작성)*

---
## 🔴 심화 문제

### 7. [실험] 클러스터링 — 선형 탐사법의 약점

선형 탐사법은 충돌이 나면 바로 옆 칸으로 가. 그러면 **데이터가 뭉치는(클러스터링)** 현상이 생겨서 탐사 횟수가 늘어나.

해시값이 몰린 키들과 고르게 퍼진 키들의 **탐사 횟수**를 직접 재봐.

In [ ]:
def probe_count(h, key):
    '''key를 찾는 데 버킷을 몇 개나 들여다봤는지'''
    hash = h.hash_value(key)
    p = h.table[hash]
    for i in range(h.capacity):
        if p.stat == Status.EMPTY:
            return i + 1
        elif p.stat == Status.OCCUPIED and p.key == key:
            return i + 1
        hash = h.rehash_value(hash)
        p = h.table[hash]
    return h.capacity

# 해시값이 전부 1로 몰리는 키들
h1 = OpenHash(13)
clustered = [1, 14, 27, 40, 53, 66]
for k in clustered:
    h1.add(k, f'v{k}')
print('몰린 키 :', clustered, '(전부 13으로 나눈 나머지가 1)')
print('탐사 횟수:', [probe_count(h1, k) for k in clustered])
print()

# 해시값이 고르게 퍼진 키들
h2 = OpenHash(13)
spread = [1, 2, 3, 4, 5, 6]
for k in spread:
    h2.add(k, f'v{k}')
print('퍼진 키 :', spread)
print('탐사 횟수:', [probe_count(h2, k) for k in spread])
print()
print('-> 몰리면 탐사 횟수가 1,2,3,4,5,6... 선형으로 증가 = O(1)이 O(n)으로 퇴화')


### 8. [디버깅] 교재 각주의 진짜 이유 🔥

교재 144p에 이런 각주가 있어:

> "만약 값이 없고 키만 있는 데이터의 해시 테이블을 사용해야 한다면 `hash.add(key, key)`처럼 키와 값에 같은 변수를 전달하면 됩니다."

왜 이런 이상한 조언이 붙었을까? 아래를 실행해서 확인해봐.

In [ ]:
h = OpenHash(13)
print('add(1, None) ->', h.add(1, None))
print('search(1)    ->', h.search(1), '  <- 저장했는데 None이 나옴')
print('add(1, "덮어쓰기") ->', h.add(1, '덮어쓰기'), '  <- 중복 키인데 True?!')
print()
h.dump()
print()
print('키 1이 버킷 1, 2에 두 개 존재!')


**(a)** 이 버그의 원인이 되는 `add`의 한 줄을 찾아봐.

**(b)** `search`와 `search_node`의 반환값 차이를 이용해서 이 버그를 고쳐봐.

In [ ]:
class FixedHash(OpenHash):
    def add(self, key, value):
        if ___ is not None:   # TODO: 어떤 함수를 써야 None 값도 구분할 수 있을까?
            return False
        hash = self.hash_value(key)
        p = self.table[hash]
        for i in range(self.capacity):
            if p.stat == Status.EMPTY or p.stat == Status.DELETED:
                self.table[hash] = Bucket(key, value, Status.OCCUPIED)
                return True
            hash = self.rehash_value(hash)
            p = self.table[hash]
        return False

h = FixedHash(13)
print('add(1, None)       ->', h.add(1, None))       # True
print('add(1, "덮어쓰기") ->', h.add(1, '덮어쓰기'))  # False 여야 정상


### 9. [설명] 체인법 vs 오픈 주소법, 뭘 언제 써?

**(a)** 8일차 체인법과 오늘 오픈 주소법을 비교하는 표를 채워봐.

| | 체인법 | 오픈 주소법 |
|---|---|---|
| 충돌 시 | ? | ? |
| 노드 구조 | key, value, **next** | key, value, **stat** |
| 적재율 1 초과 가능? | ? | ? |
| 추가 메모리 | ? | ? |
| 삭제 방식 | ? | ? |

**(b)** `remove`의 어노테이션이 이래:
```python
def remove(self, key: Any) -> int:   # 우리 코드 & 교재 090행 둘 다
    ...
    return False   # bool을 반환!
```
8일차 9번과 같은 문제야. 왜 에러가 안 나고, 뭐로 고쳐야 해?

**(c)** 오픈 주소법에서 적재율이 1에 가까워지면 무슨 일이 생겨? `add`의 `return False`(맨 끝)는 언제 실행돼?

*(여기에 답 작성)*

---
## ✅ 정답 & 해설

### R-1. 체인법 vs 오픈 주소법 (교재 150p)

- **체인법**: 해시값이 같은 `1(수연)`과 `14(민서)`를 연결하는 **연결 리스트가 버킷 1에** 매달려 있음. 14는 **버킷 1**에 있음 (리스트 안에).
- **오픈 주소법**: 나중에 추가한 `14(민서)`는 **재해시 결과 버킷 2**에 등록됨.
- **근본적 차이**: 체인법은 **버킷 밖(연결 리스트)으로 확장**해서 충돌을 흡수하고, 오픈 주소법은 **테이블 안의 다른 빈 버킷**을 찾아 들어간다. 즉 "세로로 쌓기 vs 옆으로 밀기".

---
### R-2. 체인법 remove 버그

**버그**: `pp = p`가 `if p.key == key` **앞**에 있음.

**왜 틀렸나**: `pp`는 "**바로 앞의** 노드"여야 하는데, 검사하기 전에 `pp = p`를 해버리면 `pp`와 `p`가 **같은 노드**를 가리켜. 그래서 head를 삭제할 때도 `pp is None`이 절대 참이 안 되고, `pp.next = p.next`는 `p.next = p.next`가 되어 **아무것도 안 지워져**.

**실행 결과**: `remove(14)`가 `True`를 반환하는데 버킷은 그대로 `14(B) -> 1(A)`.

**해결**: 원본처럼 `pp = p`를 루프 **맨 끝**(`p = p.next` 직전)으로 옮겨야 해.
```python
while p is not None:
    if p.key == key:
        ...
        return True
    pp = p          # 검사 후에 갱신
    p = p.next
```

**교훈**: "앞 노드를 추적하는 변수"는 **다음 칸으로 넘어가기 직전**에 갱신해야 해. 순서가 한 줄만 바뀌어도 의미가 완전히 달라져. (8일차 6번은 분기가 없어서 터졌고, 이번엔 분기는 있는데 순서가 틀려서 조용히 실패 — 더 위험한 버그야.)

---
### 1. 재해시 추적 정답

| 키 | 해시값 | 충돌? | 최종 버킷 |
|---|---|---|---|
| 1 | 1 | X | 1 |
| 5 | 5 | X | 5 |
| 10 | 10 | X | 10 |
| 12 | 12 | X | 12 |
| 14 | **1** | **O** (1이 이미 있음) | **2** (재해시 1회) |

- 14는 `14 % 13 = 1` → 버킷 1에 이미 `1(수연)`이 있어 충돌 → 재해시 `(1+1) % 13 = 2` → 버킷 2가 비어있어서 **버킷 2**에 저장. 재해시 **1번**.
- **8일차 체인법과의 차이**: 체인법에서 14는 버킷 1의 연결 리스트 안(`1 → 14 → 1`)에 있었지만, 오픈 주소법에서는 **버킷 2라는 완전히 다른 자리**를 차지해. 교재 151p 실행결과 `2 14 (민서)`와 일치.

---
### 2. 구현 정답
```python
def rehash_value(self, key):
    return (self.hash_value(key) + 1) % self.capacity

def search_node(self, key):
    hash = self.hash_value(key)
    p = self.table[hash]
    for i in range(self.capacity):
        if p.stat == Status.EMPTY:
            break
        elif p.stat == Status.OCCUPIED and p.key == key:
            return p
        hash = self.rehash_value(hash)
        p = self.table[hash]
    return None
```

**왜 `% self.capacity`가 필요한가**: 테이블 끝(버킷 12)에서 재해시하면 13이 되는데, 그건 인덱스 범위 밖이야. `% 13`을 하면 0으로 **한 바퀴 돌아와**. 즉 테이블을 **원형(circular)**으로 취급하는 거야.

---
### 3. 선형 탐사법 설명

- **왜 "선형"인가**: 충돌하면 `+1, +1, +1...`로 **한 칸씩 일정한 간격(직선적으로)** 옆으로 이동하며 빈 자리를 찾기 때문. 탐사 위치가 시작점에서 `i`번째면 `(hash + i) % capacity`라는 **1차식(선형식)**으로 표현돼.
- **`+3`으로 해도 될까**: `capacity`와 증가폭이 **서로소면 OK**(전체 버킷을 다 방문 가능), **공약수가 있으면 문제**야.
  - 예: `capacity=12`, `+3` → `0 → 3 → 6 → 9 → 0` 이렇게 4칸만 뱅뱅 돌고 나머지 8칸엔 **영원히 도달 못 해**. 테이블이 텅텅 비었는데도 `add`가 실패해.
  - `capacity=13`(소수), `+3` → 13과 3이 서로소라 13칸을 전부 방문 가능.
  - **8일차 소수 실험과 정확히 같은 원리** — capacity가 소수면 어떤 증가폭과도 대부분 서로소라 안전해.

---
### 4. `[Bucket()] * capacity` 🔥🔥🔥

**(a)** `table[0] is table[1] is table[7]` → **`True`**. 서로 다른 Bucket 객체는 **1개뿐**인데 capacity는 13. 즉 **같은 객체를 13번 참조**하는 리스트야.

**(b) 왜 우연히 동작하나**: `add`가 `self.table[hash] = Bucket(...)`로 **새 객체를 만들어 재대입**하기 때문. 재대입은 리스트의 그 칸이 **가리키는 대상만 바꾸는** 거라 공유 객체 자체는 안 건드려. `remove`도 `p.set_status(DELETED)`로 제자리 수정을 하긴 하지만, `p`는 `search_node`가 찾은 **OCCUPIED 버킷**(= add에서 새로 만든 객체)이라 공유 객체가 아니야. 그래서 공유된 EMPTY 버킷은 **한 번도 수정되지 않고** 살아남아.

**(c) 실행 결과**: `add(1, '수연')` **한 번**만 했는데 **13칸 전부** `1 (수연)`으로 채워짐! `p.set(...)`은 공유 객체를 **제자리에서 수정**하니까, 그 하나를 바꾸면 13개 칸이 전부 동시에 바뀌어 보이는 거야.

**(d) 5일차 개념 연결**: 이건 **얕은 복사(shallow copy)**와 정확히 같은 문제야. 5일차 R-2에서 확인한 것처럼 "**바깥 그릇은 새로 만들어지지만 안쪽 원소는 공유**"돼. `[Bucket()] * 13`은 `Bucket()`을 **딱 한 번만 평가**하고 그 결과(참조)를 13번 복사하는 거지, `Bucket()`을 13번 호출하는 게 아니야.

**해결**:
```python
self.table = [Bucket() for _ in range(self.capacity)]   # 내포 표기 (3일차!)
```
내포 표기는 `Bucket()`을 **매 반복마다 새로 호출**하니까 13개의 독립된 객체가 생겨.

**왜 이게 중요한가**: 지금은 우연히 동작하지만, 나중에 누가 "최적화한답시고" `add`를 `p.set(...)`으로 바꾸거나, 새 메서드에서 버킷을 제자리 수정하는 순간 **조용히 터져**. 교재 코드가 `set()` 메서드를 정의해놓고 안 쓰는 게 오히려 다행인 셈. 🎯

> 💡 같은 함정: `[[0] * 3] * 3`으로 2차원 배열 만들면 행이 전부 공유돼. `[[0] * 3 for _ in range(3)]`이 정답.

---
### 5. DELETED 필요성

**실행 결과**:
- `DELETED 사용(원본)`: 5 삭제 후에도 `search(18)` → `eighteen` ✓
- `EMPTY 사용(버그)`: 5 삭제 후 `search(18)` → **`None`** ✗

**왜**: 18은 `18 % 13 = 5`가 원래 자리인데 5가 차지하고 있어서 재해시로 버킷 6에 저장됐어. 18을 검색하려면 **버킷 5부터 시작해서 6으로 탐사**해야 해. 그런데 5를 지우면서 `EMPTY`로 만들면, `search_node`가 버킷 5에서 `EMPTY`를 보고 **즉시 `break`** → "18은 없다"고 잘못 결론.

`DELETED`는 **"여기 있던 건 지워졌지만, 나를 거쳐서 재해시된 데이터가 뒤에 있을 수 있으니 계속 가라"**는 이정표 역할이야. (교재 145p: "해시값이 같은 18을 검색할 때 해시값이 5인 데이터는 존재하지 않는다고 착각하여 검색에 실패하기 때문입니다.")

---
### 6. add가 `EMPTY or DELETED` 둘 다 받는 이유

- **search_node 관점**: `DELETED`는 "**과거에 데이터가 있었다**"는 흔적 → 이 자리를 거쳐 재해시된 데이터가 뒤에 있을 수 있음 → **지나쳐서 계속 탐사**해야 함.
- **add 관점**: `DELETED`는 "**지금은 비어 있다**" → 새 데이터를 넣어도 됨 → **재사용 가능한 자리**.

즉 같은 `DELETED`가 검색에겐 "**통과 표지판**", 추가에겐 "**빈 자리**"인 거야. 관점에 따라 의미가 다른 게 핵심.

**재사용 안 하면**: 삭제를 반복할수록 `DELETED` 자리가 계속 쌓여서, 실제로는 비어 있는데도 `add`가 `EMPTY`를 못 찾아 실패하게 돼. 테이블이 **서서히 오염**되어 결국 `add`가 `return False`(맨 끝, 테이블 가득 참)를 반환. 삭제·추가가 잦은 상황에서 치명적.

---
### 7. 클러스터링 실험 해설

**결과**:
- 몰린 키 `[1, 14, 27, 40, 53, 66]` → 탐사 횟수 `[1, 2, 3, 4, 5, 6]` — **선형 증가!**
- 퍼진 키 `[1, 2, 3, 4, 5, 6]` → 탐사 횟수 `[1, 1, 1, 1, 1, 1]` — 전부 1번

**왜**: 해시값이 전부 1로 같으니까 1, 2, 3, 4, 5, 6번 버킷이 **연달아 뭉쳐(클러스터)**. 나중에 들어온 66을 찾으려면 버킷 1부터 6까지 **6칸을 순서대로 훑어야** 해 — 이게 사실상 **선형 검색(6일차)**이야.

**핵심**: 오픈 주소법에서 클러스터가 커지면 O(1)이 **O(n)으로 퇴화**해. 이게 선형 탐사법의 약점이고, 실무에서는 **제곱 탐사법(quadratic probing)**(`+1, +4, +9...`)이나 **이중 해싱(double hashing)**(증가폭 자체를 두 번째 해시 함수로 결정)으로 클러스터를 흩뜨려.

---
### 8. 교재 각주의 진짜 이유 🔥

**(a) 원인이 되는 줄**:
```python
if self.search(key) is not None:   # <- 여기
    return False
```

`search`는 **두 가지 다른 상황에서 똑같이 `None`을 반환**해:
1. 키가 아예 없음 (검색 실패)
2. 키는 있는데 **값이 `None`** (검색 성공인데 값이 None)

그래서 `add(1, None)`으로 저장한 뒤 `add(1, '덮어쓰기')`를 하면, `search(1)`이 `None`을 반환 → "등록 안 됐네"라고 착각 → **중복 키가 두 개** 생김.

**(b) 수정**:
```python
if self.search_node(key) is not None:   # search -> search_node
    return False
```

`search_node`는 **Bucket 객체 자체**를 반환하니까, "키가 없음"일 때만 `None`이야. 값이 `None`이어도 Bucket 객체는 존재하므로 정확히 구분돼.

**교훈**: `None`을 "**없음**"의 신호(sentinel)로 쓰는데 **저장하는 값에도 `None`이 올 수 있다면**, 그 둘을 구분할 수 없게 돼. 이걸 피하려면 (1) 객체 자체를 반환하거나, (2) 별도의 `found` 플래그를 반환하거나, (3) 아예 다른 sentinel 객체를 쓰면 돼.

> 교재가 `hash.add(key, key)`를 권한 건 **버그를 고치는 대신 우회하는 방법**이야. `value`에 `None`이 절대 안 들어가게 만들어서 문제를 회피한 거지. 근본 해결은 `search_node`를 쓰는 것. (6일차 보초법에서 `-1`을 실패 신호로 쓴 것과 같은 결 — sentinel 값 설계는 항상 "그 값이 정상 데이터로도 올 수 있나?"를 따져야 해.)

---
### 9. 체인법 vs 오픈 주소법

**(a)**

| | 체인법 | 오픈 주소법 |
|---|---|---|
| 충돌 시 | 연결 리스트에 **매달기** | 재해시로 **다른 빈 버킷 찾기** |
| 노드 구조 | key, value, **next** | key, value, **stat** |
| 적재율 1 초과 가능? | **가능** (리스트는 무한히 길어짐) | **불가능** (버킷 수가 상한) |
| 추가 메모리 | 노드마다 next 포인터 | 없음 (테이블만) |
| 삭제 방식 | 노드를 **실제로 제거** | `DELETED`로 **표시만** |

**(b) `-> int` 어노테이션**: 파이썬 어노테이션은 **강제되지 않는 힌트**라서, `-> int`라고 써놓고 `bool`을 반환해도 인터프리터가 검사를 안 해. 그냥 `__annotations__`에 메타데이터로 저장될 뿐. **`-> bool`로 고쳐야** 맞아. (8일차 `__init__ -> int` 문제와 동일 — 교재가 두 번 다 실수한 셈. 틀린 어노테이션은 "동작은 하지만 거짓말하는 문서"라 없느니만 못해.)

> 참고: 파이썬에서 `bool`은 `int`의 **서브클래스**라서 `isinstance(True, int)`가 `True`야. 그래서 `-> int`가 아주 틀린 건 아니라고 우길 수도 있지만... 의도를 표현하는 게 어노테이션의 목적이니 `-> bool`이 옳아.

**(c) 적재율이 1에 가까워지면**:
- 빈 버킷이 거의 없어서 재해시를 **수십 번** 반복해야 자리를 찾음 → 클러스터가 극심해져 **O(n)으로 퇴화**
- 테이블이 **완전히 가득 차면**, `add`의 `for i in range(self.capacity)` 루프가 capacity번 돌고도 자리를 못 찾아서 **맨 끝 `return False`**가 실행돼 → 추가 실패
- 체인법은 적재율이 1을 넘어도 리스트가 길어질 뿐 **저장은 되지만**, 오픈 주소법은 **물리적으로 못 넣어**. 이게 가장 큰 차이.
- 실무 해시맵은 적재율이 임계값(보통 0.5~0.75)을 넘으면 **테이블을 키우고 전부 재해싱**(rehashing)해.

---

## 📌 핵심 3줄 요약

1. **오픈 주소법**은 충돌 시 **재해시로 빈 버킷을 찾아** 테이블 **안에서** 해결한다 (체인법은 리스트로 밖에 매달기). 한 칸씩 옆으로 가는 방식이 **선형 탐사법**.
2. 버킷에 **3가지 속성**(OCCUPIED/EMPTY/DELETED)이 필요한 이유는 삭제 때문 — `DELETED`는 검색에겐 "**계속 가라**", 추가에겐 "**여기 써도 돼**"라는 이중 신호다.
3. `[Bucket()] * capacity`는 **같은 객체를 13번 참조**하는 5일차 얕은 복사 함정이다. 지금은 `add`가 재대입해서 우연히 살아있지만, 제자리 수정하는 순간 터진다 → `[Bucket() for _ in range(capacity)]`가 정답.

## 🗂️ 스터디 진행 가이드
- 🟢 (1~3번): 전원 필수. 1번 재해시 손추적은 오픈 주소법 이해의 출발점
- 🟡 (4~6번): 팀 목표선.
  - **4번은 오늘의 최고 문제** 🔥 — 교재 코드에 들어있는 진짜 시한폭탄. 5일차 얕은 복사가 왜 중요했는지 여기서 회수됨
  - 5번 DELETED는 "왜 상태가 3개나 필요한가"의 답. 이거 모르면 오픈 주소법 이해한 게 아님
- 🔴 (7~9번): 도전. **8번은 교재 각주의 숨은 이유**를 파헤치는 문제라 재밌을 거야
- **금요일 코딩테스트 범위**: 🟢🟡 (1~6번)

## 🔗 03장 검색 알고리즘 완주! 총정리

| | 선형 검색 | 이진 검색 | 체인법 | 오픈 주소법 |
|---|---|---|---|---|
| **일차** | 6일차 | 7일차 | 8일차 | **9일차** |
| **전제** | 없음 | **정렬** 필수 | 없음 | 없음 |
| **검색** | O(n) | O(log n) | O(1)* | O(1)* |
| **추가** | O(1) | O(n) | O(1)* | O(1)* |
| **삭제** | O(n) | O(n) | O(1)* | O(1)* |
| **충돌 대처** | — | — | 연결 리스트 | 재해시 |
| **추가 메모리** | 없음 | 없음 | next 포인터 | 없음 |
| **약점** | 느림 | 추가/삭제 비쌈 | 포인터 오버헤드 | **클러스터링** |

\* 충돌이 적을 때. 클러스터/체인이 길어지면 O(n)으로 퇴화.

### 🎓 03장에서 회수된 이전 개념들
- **6일차 선형 검색** → 체인법의 연결 리스트 순회, 오픈 주소법의 클러스터 탐사
- **7일차 이진 검색** → 정렬 배열 삽입 O(n)이 해시법의 동기
- **5일차 얕은 복사** → `[Bucket()] * capacity` 함정 (9일차 4번) 🎯
- **5일차 소수 √n** → 해시 테이블 크기가 소수인 이유 (8일차 7번), 재해시 증가폭과 서로소 (9일차 3번)
- **3일차 `is` vs `==`** → `p is None`, `table[0] is table[1]`
- **3일차 내포 표기** → `[Bucket() for _ in range(n)]`이 정답인 이유
- **4일차 어노테이션** → `-> int` vs `-> bool` (8, 9일차)

> **다음 진도**: 04장 스택과 큐 (04-1 스택이란? / 04-2 큐란?)
